# Titanic Survival Prediction - Section A: Problem Definition & Data Preparation

**Prediction objective:** Predict whether a passenger survived the Titanic sinking, using their demographic and travel details (class, sex, age, family size, fare, port of embarkation).

This notebook produces the cleaned, split datasets (`X_train`, `X_test`, `y_train`, `y_test`) used by the modelling notebooks (`from_scratch_models.ipynb` and `C_library_models.ipynb`).

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


np.random.seed(42)

## 1. Load the data

In [12]:
df = pd.read_csv('Titanic-Dataset.csv')
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


## 2. Drop identifiers and free-text columns

`PassengerId`, `Name`, and `Ticket` are unique identifiers / free text with no direct predictive value on their own.
`Cabin` is missing for ~77% of passengers, so it's dropped rather than imputed.

In [14]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## 3. Simple imputation

- `Age`: fill missing values with the **median** age.
- `Embarked`: fill the 2 missing values with the **mode** (most common port).

In [15]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


## 4. Encode categorical variables

- `Sex`: binary map (`male` -> 0, `female` -> 1).
- `Embarked`: one-hot encode (dropping the first category to avoid redundancy). Embarked_C = 1 when Embarked_Q = 0 & Embarked_S = 0

In [16]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], prefix='Embarked', drop_first=True)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,False,True
1,1,1,1,38.0,1,0,71.2833,False,False
2,1,3,1,26.0,0,0,7.9250,False,True
3,1,1,1,35.0,1,0,53.1000,False,True
4,0,3,0,35.0,0,0,8.0500,False,True


## 5. Train/test split (stratified, 80/20)

In [17]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print('X_train:', X_train.shape, ' X_test:', X_test.shape)
print('\nSurvival rate (train):', y_train.mean().round(3))
print('Survival rate (test): ', y_test.mean().round(3))

X_train: (712, 8)  X_test: (179, 8)

Survival rate (train): 0.383
Survival rate (test):  0.385


## 6. Save the prepared datasets

In [18]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('Saved X_train.csv, X_test.csv, y_train.csv, y_test.csv')

Saved X_train.csv, X_test.csv, y_train.csv, y_test.csv
